In [1]:
import requests

base_url = "http://localhost:8002"
endpoint = f"{base_url}/api/events/"
print(endpoint)

http://localhost:8002/api/events/


In [2]:
# health check first
r = requests.get(f"{base_url}/healthz")
print(r.status_code, r.json())

200 {'status': 'ok'}


In [3]:
# create
r = requests.post(endpoint, json={"page": "/about", "description": "first event"})
print(r.status_code)
created = r.json()
created

201


{'created_at': '2026-09-12T05:33:22.844637Z',
 'id': 11,
 'description': 'first event',
 'updated_at': '2026-09-12T05:33:22.844644Z',
 'page': '/about'}

In [4]:
event_id = created["id"]
print("event_id:", event_id)
print("created_at:", created["created_at"])
print("updated_at:", created["updated_at"])

event_id: 11
created_at: 2026-09-12T05:33:22.844637Z
updated_at: 2026-09-12T05:33:22.844644Z


In [5]:
# make a few more so the list has something to order
for page in ["/pricing", "/contact", "/blog"]:
    requests.post(endpoint, json={"page": page})

r = requests.get(endpoint, params={"limit": 3})
data = r.json()
print(r.status_code, "count:", data["count"])
for row in data["results"]:
    print(row["id"], row["page"])

200 count: 3
14 /blog
13 /contact
12 /pricing


In [11]:
# no limit param - endpoint defaults to 10, newest first
r = requests.get(endpoint)
data = r.json()
print(r.status_code, "count:", data["count"])

for row in data["results"]:
    print(f'{row["id"]:>4}  {row["page"]:<12} {row["created_at"]}')


200 count: 10
  18  /blog        2026-09-12T05:35:43.363140Z
  17  /contact     2026-09-12T05:35:43.356251Z
  16  /pricing     2026-09-12T05:35:43.349441Z
  14  /blog        2026-09-12T05:33:39.029399Z
  13  /contact     2026-09-12T05:33:39.022070Z
  12  /pricing     2026-09-12T05:33:38.977031Z
  10  /blog        2026-09-12T05:31:54.596501Z
   9  /contact     2026-09-12T05:31:54.590944Z
   8  /pricing     2026-09-12T05:31:54.584405Z
   6  /blog        2026-09-12T05:24:01.581005Z


In [6]:
# detail
r = requests.get(f"{endpoint}{event_id}")
print(r.status_code)
r.json()

200


{'created_at': '2026-09-12T05:33:22.844637Z',
 'id': 11,
 'description': 'first event',
 'updated_at': '2026-09-12T05:33:22.844644Z',
 'page': '/about'}

In [7]:
# missing id should 404
r = requests.get(f"{endpoint}999999")
print(r.status_code, r.json())

404 {'detail': 'Event not found'}


In [8]:
# update - updated_at should move, created_at should not
r = requests.put(f"{endpoint}{event_id}", json={"description": "updated!"})
print(r.status_code)
updated = r.json()

print("description:", updated["description"])
print("created_at same:", updated["created_at"] == created["created_at"])
print("updated_at changed:", updated["updated_at"] != created["updated_at"])

200
description: updated!
created_at same: True
updated_at changed: True


In [9]:
# delete returns 204 with no body
r = requests.delete(f"{endpoint}{event_id}")
print(r.status_code, repr(r.text))

204 ''


In [10]:
# gone now
r = requests.get(f"{endpoint}{event_id}")
print(r.status_code, r.json())

404 {'detail': 'Event not found'}
